
<img alt="Zero Grad" src="https://imgtr.ee/images/2023/06/20/ZrpSs.png" >

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'presentation'
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import sys
sys.path.append('/content/drive/MyDrive/Grokking Machine Learning/03- Python Machine Learning/W17')

from pca_module import standardize_data, compute_covariance_matrix, calculate_eigenvectors, select_principal_components, transform_data

In [ ]:
df = pd.DataFrame({'x1': [2, 3, 4, -2, -4, -3], 'x2': [1, 5, 3, -2, -3, -5]})
df

,x1,x2
0,2,1
1,3,5
2,4,3
3,-2,-2
4,-4,-3
5,-3,-5


In [ ]:
fig = px.scatter(df, x='x1', y='x2', width=800, height=500)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.show()


In [ ]:
X = df.values

# Step 1: Standardize the data
X_std = standardize_data(X)

# Step 2: Compute the covariance matrix
covariance_matrix = compute_covariance_matrix(X_std)

# Step 3: Calculate the eigenvectors and eigenvalues
eigenvalues, eigenvectors = calculate_eigenvectors(covariance_matrix)

# Step 4: Select the principal components
k = 2  # Choose the desired number of principal components
selected_eigenvectors = select_principal_components(eigenvalues, eigenvectors, k)

# Step 5: Transform the data
transformed_data = transform_data(X_std, selected_eigenvectors)


In [ ]:
X_std

array([[ 0.64326752,  0.33485541],
       [ 0.96490128,  1.48293111],
       [ 1.28653504,  0.90889326],
       [-0.64326752, -0.52620136],
       [-1.28653504, -0.81322028],
       [-0.96490128, -1.38725813]])

In [ ]:
covariance_matrix

array([[1.2       , 1.10777971],
       [1.10777971, 1.2       ]])

In [ ]:
# The new unit vectors are the eigenvectors of the covariance matrix
eigenvectors

array([[ 0.70710678, -0.70710678],
       [ 0.70710678,  0.70710678]])

In [ ]:
pc1 = eigenvectors[:, 0]
pc2 = eigenvectors[:, 1]

print('First principal component:', pc1)
print('Second principal component:', pc2)

First principal component: [0.70710678 0.70710678]
Second principal component: [-0.70710678  0.70710678]


In [ ]:
# The eigenvalues of the covariance matrix represent the variance of the data
eigenvalues

array([2.30777971, 0.09222029])

In [ ]:
singular_values = np.sqrt(eigenvalues)
singular_values

array([1.51913782, 0.30367794])

In [ ]:
transformed_data

array([[ 0.69163736, -0.21808029],
       [ 1.73087888,  0.3663024 ],
       [ 1.55240224, -0.26703307],
       [-0.82693938,  0.08277828],
       [-1.48475123,  0.33468407],
       [-1.66322787, -0.29865139]])

# PCA from Sklearn

In [ ]:
# PCA from sklearn
scaler = StandardScaler()
pca_sklearn = PCA(n_components=2)

pca_pipeline = Pipeline([('scaler', scaler), ('pca', pca_sklearn)])
transformed_data_sklearn = pca_pipeline.fit_transform(X)


In [ ]:
# Compare the results

print("PCA from scratch: \n", transformed_data)
print("\nPCA from sklearn: \n", transformed_data_sklearn)


PCA from scratch: 
 [[ 0.69163736 -0.21808029]
 [ 1.73087888  0.3663024 ]
 [ 1.55240224 -0.26703307]
 [-0.82693938  0.08277828]
 [-1.48475123  0.33468407]
 [-1.66322787 -0.29865139]]

PCA from sklearn: 
 [[ 0.69163736 -0.21808029]
 [ 1.73087888  0.3663024 ]
 [ 1.55240224 -0.26703307]
 [-0.82693938  0.08277828]
 [-1.48475123  0.33468407]
 [-1.66322787 -0.29865139]]


In [ ]:
np.allclose(transformed_data, transformed_data_sklearn)

True

In [ ]:
np.allclose(eigenvalues, pca_sklearn.explained_variance_)

True

In [ ]:
np.allclose(eigenvectors, pca_sklearn.components_.T)

True

# Reconstruction

In [ ]:
# Project the data onto the first principal component
df_projected = np.dot(df, eigenvectors[:, 0])

# inverse transform the projected data
df_reconstructed = np.dot(df_projected.reshape(-1, 1), eigenvectors[:, 0].reshape(1, -1))
df_reconstructed

array([[ 1.5,  1.5],
       [ 4. ,  4. ],
       [ 3.5,  3.5],
       [-2. , -2. ],
       [-3.5, -3.5],
       [-4. , -4. ]])

In [ ]:
# Plot the original data and the reconstructed data
fig = px.scatter(df, x='x1', y='x2', width=800, height=500)
fig.add_trace(go.Scatter(x=df_reconstructed[:, 0]*1.2, y=df_reconstructed[:, 1]*1.2, mode='lines', name='Reconstructed'))
fig.add_trace(go.Scatter(x=df_reconstructed[:, 0], y=df_reconstructed[:, 1], mode='markers', name='Reconstructed'))

fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.show()

In [ ]:
# Reconstruction error
reconstruction_error = np.sum((X - df_reconstructed)**2)
reconstruction_error

5.5

In [ ]:
X

array([[ 2,  1],
       [ 3,  5],
       [ 4,  3],
       [-2, -2],
       [-4, -3],
       [-3, -5]])

In [ ]:
df_reconstructed

array([[ 1.5,  1.5],
       [ 4. ,  4. ],
       [ 3.5,  3.5],
       [-2. , -2. ],
       [-3.5, -3.5],
       [-4. , -4. ]])

In [ ]:
pca_pipeline.inverse_transform(transformed_data_sklearn)

array([[ 2.,  1.],
       [ 3.,  5.],
       [ 4.,  3.],
       [-2., -2.],
       [-4., -3.],
       [-3., -5.]])

In [ ]:
pca_1 = Pipeline([('scaler', scaler), ('pca', PCA(n_components=1))])
pc1 = pca_1.fit_transform(X)

pc1

array([[ 0.69163736],
       [ 1.73087888],
       [ 1.55240224],
       [-0.82693938],
       [-1.48475123],
       [-1.66322787]])

In [ ]:
X_reconstructed = pca_1.inverse_transform(pc1)
X_reconstructed

array([[ 1.52055389,  1.5372679 ],
       [ 3.80531009,  4.09756852],
       [ 3.41293199,  3.65786914],
       [-1.81801326, -2.20393457],
       [-3.26420231, -3.8245358 ],
       [-3.65658041, -4.26423518]])

In [ ]:
X

array([[ 2,  1],
       [ 3,  5],
       [ 4,  3],
       [-2, -2],
       [-4, -3],
       [-3, -5]])

In [ ]:
# Reconstruction error
reconstruction_error = np.sum((X - X_reconstructed)**2)
reconstruction_error

5.027286645961915

In [ ]:
# Cumulative explained variance
px.bar(x=['PC1', 'PC2'], y=pca_sklearn.explained_variance_ratio_, width=800, height=500)

In [ ]:
# Cumulative explained variance
px.area(x=range(1, 3), y=np.cumsum(pca_sklearn.explained_variance_ratio_), width=800, height=500)